# Experiment: Inspect MultiLepPAT GEN-RECO Matching

This notebook inspects a `MultiLepPAT` ntuple with emphasis on three questions:

1. Is the GEN-level event content kept even when reco selection is sparse?
2. How often do muons use PAT GEN refs versus the chi2 fallback?
3. What decay signatures are visible from the stored `MC_GenPart_*` flat collection?

The notebook is written to work on the original user ntuple by default, and falls back to the documented validation ntuple produced during the GEN-matching fix validation.

## Notebook Outline

- Configure the ntuple path and open `mkcands/X_data`.
- Inspect branch inventory for GEN and matching content.
- Summarize event-level GEN retention and muon match provenance.
- Inspect one event in detail, joining reco muons to matched GEN particles.
- Scan approximate mother-daughter decay signatures from `MC_GenPart_pdgId` and `MC_GenPart_motherPdgId`.

Limitation: the ntuple stores `motherPdgId`, not a full mother index, so the decay-chain view is pattern-based rather than a full per-particle ancestry graph.

In [ ]:
from pathlib import Path
from collections import Counter
from math import hypot, isfinite

import ROOT
from IPython.display import display

try:
    import pandas as pd
except ImportError:
    pd = None

ROOT.gROOT.SetBatch(True)

PRIMARY_NTUPLE = Path("/eos/user/c/chiw/JpsiJpsiPhi/CMSSW_15_0_15_JpsiJpsiPhi_refactor/src/mymultilep_JpsiUpsPhi_mcRun2022_TPS.root")
VALIDATION_NTUPLE = Path("/tmp/chiw_test_JpsiJpsiPhi_impl_20260330_numEvent5.root")
TREE_PATH = "mkcands/X_data"

NTUPLE_PATH = PRIMARY_NTUPLE if PRIMARY_NTUPLE.exists() else VALIDATION_NTUPLE
if not NTUPLE_PATH.exists():
    raise FileNotFoundError("Set NTUPLE_PATH to a valid MultiLepPAT ntuple before running the notebook.")

root_file = ROOT.TFile.Open(str(NTUPLE_PATH), "READ")
tree = root_file.Get(TREE_PATH)
if not tree:
    raise RuntimeError(f"Could not find {TREE_PATH} in {NTUPLE_PATH}")

print(f"Using ntuple: {NTUPLE_PATH}")
print(f"Entries in {TREE_PATH}: {tree.GetEntries()}")


In [ ]:
branches = [branch.GetName() for branch in tree.GetListOfBranches()]
focus_branches = [
    name for name in branches
    if name.startswith("MC_GenPart")
    or name.startswith("muGenMatch")
    or name in {"Jpsi_1_mu_1_Idx", "Jpsi_1_mu_2_Idx", "Jpsi_2_mu_1_Idx", "Jpsi_2_mu_2_Idx", "Ups_mu_1_Idx", "Ups_mu_2_Idx"}
]

for name in focus_branches:
    print(name)


In [ ]:
PDG_LABELS = {
    443: "J/psi",
    -443: "J/psi",
    553: "Upsilon",
    -553: "Upsilon",
    333: "phi",
    -333: "phi",
    13: "mu-",
    -13: "mu+",
    321: "K+",
    -321: "K-",
}

MATCH_SOURCE_LABELS = {0: "unmatched", 1: "patRef", 2: "chi2Fallback"}

def pdg_label(pdg_id):
    return PDG_LABELS.get(int(pdg_id), str(int(pdg_id)))

def as_list(vec):
    return list(vec) if vec is not None else []

def maybe_frame(rows):
    if pd is not None:
        return pd.DataFrame(rows)
    return rows

def event_snapshot(tree, entry):
    tree.GetEntry(entry)

    gen_rows = []
    for idx, pdg_id in enumerate(as_list(tree.MC_GenPart_pdgId)):
        gen_rows.append({
            "gen_idx": idx,
            "handle_index": int(tree.MC_GenPart_handleIndex[idx]) if hasattr(tree, "MC_GenPart_handleIndex") else -1,
            "pdg_id": int(pdg_id),
            "particle": pdg_label(pdg_id),
            "status": int(tree.MC_GenPart_status[idx]),
            "mother_pdg_id": int(tree.MC_GenPart_motherPdgId[idx]),
            "mother": pdg_label(tree.MC_GenPart_motherPdgId[idx]),
            "pt": float(tree.MC_GenPart_pt[idx]),
            "eta": float(tree.MC_GenPart_eta[idx]),
            "phi": float(tree.MC_GenPart_phi[idx]),
            "mass": float(tree.MC_GenPart_mass[idx]),
        })

    mu_rows = []
    mu_px = as_list(tree.muPx)
    mu_py = as_list(tree.muPy)
    mu_pz = as_list(tree.muPz)
    mu_charge = as_list(tree.muCharge)
    mu_gen_idx = as_list(tree.muGenMatchIdx) if hasattr(tree, "muGenMatchIdx") else [-1] * len(mu_px)
    mu_gen_source = as_list(tree.muGenMatchSource) if hasattr(tree, "muGenMatchSource") else [0] * len(mu_px)
    mu_gen_chi2 = as_list(tree.muGenMatchChi2) if hasattr(tree, "muGenMatchChi2") else [-1.0] * len(mu_px)

    for idx, px in enumerate(mu_px):
        match_idx = int(mu_gen_idx[idx]) if idx < len(mu_gen_idx) else -1
        matched_gen = gen_rows[match_idx] if 0 <= match_idx < len(gen_rows) else None
        mu_rows.append({
            "mu_idx": idx,
            "charge": int(mu_charge[idx]),
            "pt": hypot(float(px), float(mu_py[idx])),
            "pz": float(mu_pz[idx]),
            "match_idx": match_idx,
            "match_source": MATCH_SOURCE_LABELS.get(int(mu_gen_source[idx]), int(mu_gen_source[idx])),
            "match_chi2": float(mu_gen_chi2[idx]) if idx < len(mu_gen_chi2) and isfinite(float(mu_gen_chi2[idx])) else None,
            "matched_pdg_id": matched_gen["pdg_id"] if matched_gen else None,
            "matched_particle": matched_gen["particle"] if matched_gen else None,
            "matched_mother": matched_gen["mother"] if matched_gen else None,
            "matched_gen_pt": matched_gen["pt"] if matched_gen else None,
        })

    return {
        "entry": entry,
        "run": int(tree.runNum),
        "lumi": int(tree.lumiNum),
        "event": int(tree.evtNum),
        "nMu": int(tree.nMu),
        "gen_rows": gen_rows,
        "mu_rows": mu_rows,
    }


In [ ]:
summary = {
    "entries": int(tree.GetEntries()),
    "events_gen_nonempty": 0,
    "events_with_muons": 0,
    "events_with_patref_match": 0,
    "events_with_chi2_fallback": 0,
    "total_muons": 0,
    "total_matched_muons": 0,
    "total_patref_matches": 0,
    "total_chi2_fallback_matches": 0,
}

for entry in range(tree.GetEntries()):
    snap = event_snapshot(tree, entry)
    gen_rows = snap["gen_rows"]
    mu_rows = snap["mu_rows"]
    if gen_rows:
        summary["events_gen_nonempty"] += 1
    if mu_rows:
        summary["events_with_muons"] += 1
    if any(row["match_source"] == "patRef" for row in mu_rows):
        summary["events_with_patref_match"] += 1
    if any(row["match_source"] == "chi2Fallback" for row in mu_rows):
        summary["events_with_chi2_fallback"] += 1
    summary["total_muons"] += len(mu_rows)
    summary["total_matched_muons"] += sum(row["match_idx"] >= 0 for row in mu_rows)
    summary["total_patref_matches"] += sum(row["match_source"] == "patRef" for row in mu_rows)
    summary["total_chi2_fallback_matches"] += sum(row["match_source"] == "chi2Fallback" for row in mu_rows)

maybe_frame([summary])


## Event Browser

Pick an event index and inspect the reco muons, their match provenance, and the stored flat GEN collection for that event.

For the decay view, remember that `motherPdgId` is enough to identify signatures like `J/psi -> mu+ mu-` or `phi -> K+ K-`, but not enough to rebuild a full graph when several mothers of the same PDG id are present in one event.

In [ ]:
ENTRY = 0
snap = event_snapshot(tree, ENTRY)
print({key: snap[key] for key in ["entry", "run", "lumi", "event", "nMu"]})
display(maybe_frame(snap["mu_rows"]))
display(maybe_frame(snap["gen_rows"]))


In [ ]:
def candidate_muon_match_rows(tree, entry):
    snap = event_snapshot(tree, entry)
    tree.GetEntry(entry)
    candidate_specs = [
        ("Jpsi_1", getattr(tree, "Jpsi_1_mu_1_Idx", []), getattr(tree, "Jpsi_1_mu_2_Idx", [])),
        ("Jpsi_2", getattr(tree, "Jpsi_2_mu_1_Idx", []), getattr(tree, "Jpsi_2_mu_2_Idx", [])),
        ("Ups", getattr(tree, "Ups_mu_1_Idx", []), getattr(tree, "Ups_mu_2_Idx", [])),
    ]
    rows = []
    for label, idx1_vec, idx2_vec in candidate_specs:
        idx1_list = as_list(idx1_vec)
        idx2_list = as_list(idx2_vec)
        for cand_idx, (mu1_idx, mu2_idx) in enumerate(zip(idx1_list, idx2_list)):
            for role, mu_idx in (("daughter_1", int(mu1_idx)), ("daughter_2", int(mu2_idx))):
                if not (0 <= mu_idx < len(snap["mu_rows"])):
                    continue
                row = dict(snap["mu_rows"][mu_idx])
                row.update({
                    "candidate": label,
                    "candidate_index": cand_idx,
                    "role": role,
                })
                rows.append(row)
    return rows

display(maybe_frame(candidate_muon_match_rows(tree, ENTRY)))


In [ ]:
def scan_decay_signatures(tree, max_events=None):
    pair_counter = Counter()
    event_counter = Counter()
    n_entries = tree.GetEntries() if max_events is None else min(tree.GetEntries(), max_events)

    for entry in range(n_entries):
        snap = event_snapshot(tree, entry)
        pairs_in_event = set()
        for row in snap["gen_rows"]:
            key = (int(row["mother_pdg_id"]), int(row["pdg_id"]))
            pair_counter[key] += 1
            pairs_in_event.add(key)
        for key in pairs_in_event:
            event_counter[key] += 1

    rows = []
    for (mother, daughter), count in pair_counter.most_common():
        if abs(mother) not in {333, 443, 553} and abs(daughter) not in {13, 321, 333, 443, 553}:
            continue
        rows.append({
            "mother_pdg_id": mother,
            "mother": pdg_label(mother),
            "daughter_pdg_id": daughter,
            "daughter": pdg_label(daughter),
            "daughter_occurrences": count,
            "events_with_signature": event_counter[(mother, daughter)],
        })
    return rows

display(maybe_frame(scan_decay_signatures(tree, max_events=500)))


In [ ]:
def match_source_breakdown(tree, max_events=None):
    source_counter = Counter()
    chi2_values = []
    n_entries = tree.GetEntries() if max_events is None else min(tree.GetEntries(), max_events)

    for entry in range(n_entries):
        snap = event_snapshot(tree, entry)
        for row in snap["mu_rows"]:
            source_counter[row["match_source"]] += 1
            if row["match_source"] == "chi2Fallback" and row["match_chi2"] is not None:
                chi2_values.append(row["match_chi2"])

    rows = [{"source": key, "count": value} for key, value in sorted(source_counter.items())]
    display(maybe_frame(rows))
    if chi2_values:
        print({
            "fallback_matches": len(chi2_values),
            "chi2_min": min(chi2_values),
            "chi2_mean": sum(chi2_values) / len(chi2_values),
            "chi2_max": max(chi2_values),
        })

match_source_breakdown(tree)


## Suggested Follow-ups

- Change `ENTRY` to inspect specific events with large GEN collections or fallback matches.
- Filter `scan_decay_signatures` to one mother PDG id at a time when comparing J/psi, Upsilon, and phi content.
- If exact ancestry becomes important, extend the ntuple with a true mother index in addition to `motherPdgId`.